<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Playground Series - Season 4 Episode 7: Binary Classification of Insurance Cross Selling</b></div>

#### Note: Please dont copy the parameters used to train the model as I have changed it of course because afterall it is a competition notebook so can't reveal the true parameter value I used (sorry)

### Information about features:
#### The Kaggle Playground Series dataset includes the following features:

- **id:** A unique identifier for each student.
- **Gender:** The gender of the customer.
- **Age:** The age of the customer.
- **Driving_License:** Indicates whether the customer has a driver's license (1 = yes, 0 = no).
- **Region_Code:** A unique code representing the customer's region.
- **Previously_Insured:** Indicates whether the customer already has vehicle insurance (1 = yes, 0 = no).
- **Vehicle_Age:** The age of the vehicle.
- **Vehicle_Damage:** Indicates if the customer has had vehicle damage in the past (1 = yes, 0 = no).
- **Annual_Premium:** The annual premium amount the customer needs to pay.
- **Policy_Sales_Channel:** An anonymized code for various customer outreach channels, such as agents, email, phone calls, etc.
- **Vintage:** The number of days the customer has been associated with the insurance company.
- **Response:** The target variable indicating whether the customer is interested (1 = Interested, 0 = Not Interested).

In [ ]:
# this is one library which is used for auto eda just like autoviz or pandas profiling but this is light weight so we gonna use it for some tasks below because already the data is huge so We cant have more than 100MB output cell
pip install klib

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Reading the dataset and getting some of the basic information</b></div>

In [ ]:
import pandas as pd
train_df = pd.read_csv('/kaggle/input/playground-series-s4e7/train.csv', index_col=[0])
test = pd.read_csv('/kaggle/input/playground-series-s4e7/test.csv', index_col=[0])

train_df.head()

In [ ]:
print(f'train_df dataFrame size: {train_df.shape}')
print(f'test dataFrame size: {test.shape}')

#### Ah thats like a huge dataset so our code is gonna take a lot of time to run :') but anyways 

In [ ]:
print(f'Number of missing values in train_df:\n{train_df.isna().sum()}')

In [ ]:
print(f'Number of missing values in test_df:\n{test.isna().sum()}')

In [ ]:
train_df.describe()

In [ ]:
categorical_columns = train_df.select_dtypes(include=['object']).columns
unique_counts = train_df[categorical_columns].nunique()
print(unique_counts)

In [ ]:
train_df.info()

In [ ]:
train_df.Gender.value_counts(normalize=True)

In [ ]:
train_df.Vehicle_Damage.value_counts(normalize=True)

In [ ]:
train_df.Vehicle_Age.value_counts(normalize=True)

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Sneak peek into the categorical features: EDA</b></div>

In [ ]:
import numpy as np
import pandas as pd
import klib

#### This is where we use klib library which makes things little easy so in the below cell we look into categorical columns

In [ ]:
klib.cat_plot(train_df)

## Insights from the Plots

1. **Gender Distribution**:
   - There are more male participants (54.1%) than female participants (45.9%).

2. **Vehicle Age Distribution**:
   - The majority of the vehicles fall into the "1-2 Year" category (95.8%).
   - A smaller portion is less than 1 year old (4.2%).
   - There are no vehicles older than 2 years represented in the data.

3. **Vehicle Damage History**:
   - The data is almost evenly split between vehicles that have damage (50.3%) and those that do not (49.7%).

### Additional Observations

- The categorical data plot indicates a balanced distribution in terms of vehicle damage history, which suggests that the dataset could be well-suited for studies involving the impact of vehicle damage on other variables.
- The gender distribution is slightly skewed towards males, which might need to be considered in analyses to ensure gender balance or account for any gender bias.
- The dominance of the "1-2 Year" category in vehicle age suggests that most vehicles in the dataset are relatively new, which could influence analyses related to vehicle performance, maintenance, and value depreciation over time.

In [ ]:
klib.corr_plot(train_df, target='Response')

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Sneak peek into the kernel density estimates (KDE) using klib : EDA</b></div>

In [ ]:
klib.dist_plot(train_df['Age'])
klib.dist_plot(train_df['Region_Code'])
klib.dist_plot(train_df['Annual_Premium'])
klib.dist_plot(train_df['Policy_Sales_Channel'])
klib.dist_plot(train_df['Vintage'])

In [ ]:
klib.dist_plot(test_df['Age'])
klib.dist_plot(test_df['Region_Code'])
klib.dist_plot(test_df['Annual_Premium'])
klib.dist_plot(test_df['Policy_Sales_Channel'])
klib.dist_plot(test_df['Vintage'])

### Insights from the kernel density estimates (KDE) 

1. **Age Distribution:**
   - **Mean Age:** 38.38 years
   - **Standard Deviation:** 14.99 years
   - **Skewness:** 0.64 (right-skewed distribution)
   - **Kurtosis:** -0.62 (platykurtic, indicating a flatter distribution)
   - The distribution has a peak around 20 years, with another smaller peak around 45 years.

2. **Region Code Distribution:**
   - **Mean Region Code:** 26.42
   - **Standard Deviation:** 12.99
   - **Skewness:** -0.13 (approximately symmetrical)
   - **Kurtosis:** -0.80 (platykurtic)
   - The distribution shows several peaks, notably around region codes 10, 20, and 30, indicating certain regions are more common.

3. **Annual Premium Distribution:**
   - **Mean Annual Premium:** 30,461.37
   - **Standard Deviation:** 16,454.75
   - **Skewness:** 0.78 (right-skewed)
   - **Kurtosis:** 24.60 (leptokurtic, indicating a peaked distribution with heavy tails)
   - The distribution shows a primary peak around 20,000, with another smaller peak around 30,000.

4. **Policy Sales Channel Distribution:**
   - **Mean Policy Sales Channel:** 112.43
   - **Standard Deviation:** 54.04
   - **Skewness:** -0.92 (left-skewed)
   - **Kurtosis:** -0.95 (platykurtic)
   - The distribution shows multiple peaks around policy sales channels 30, 80, and 150, indicating varying popularity among sales channels.

5. **Vintage (tenure) Distribution:**
   - **Mean Vintage:** 163.90 days
   - **Standard Deviation:** 79.98 days
   - **Skewness:** -0.11 (approximately symmetrical)
   - **Kurtosis:** -1.11 (platykurtic)
   - The distribution is fairly uniform with slight peaks around 50, 150, and 250 days, indicating varying lengths of tenure among customers.

### General Observations:
- The **Age** and **Annual Premium** distributions show significant right skewness, indicating a higher number of younger customers and lower premium values, with fewer older customers and higher premiums.
- The **Region Code** and **Policy Sales Channel** distributions show multiple modes, suggesting significant regional and channel-specific variations in the dataset.
- The **Vintage** distribution is relatively uniform, indicating a wide range of customer tenure lengths.

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Converting categorical features to numerical</b></div>

In [ ]:
def convert_vehicle_age(age_str):
    if isinstance(age_str, str):
        if '< 1 Year' in age_str:
            return 0
        elif '1-2 Year' in age_str:
            return 1
        elif '> 2 Years' in age_str:
            return 2
    return None

train_df['Vehicle_Age'] = train_df['Vehicle_Age'].apply(convert_vehicle_age)
train_df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

train_df['Gender'] = label_encoder.fit_transform(train_df['Gender'])
test_df['Gender'] = label_encoder.fit_transform(test_df['Gender'])
train_df['Vehicle_Damage'] = label_encoder.fit_transform(train_df['Vehicle_Damage'])
test_df['Vehicle_Damage'] = label_encoder.fit_transform(test_df['Vehicle_Damage'])
train_df.head()

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Correlation between Input features and output features</b></div>

In [ ]:
train_df.corr()['Response']

In [ ]:
import seaborn as sns  
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
fig=px.imshow(train_df.corr(),text_auto=True, template='plotly_dark', color_continuous_scale=px.colors.sequential.Blues, aspect='auto',title='<b>Correlation matrix')
fig.update_layout(title_x=0.5)
fig.show()

### Key Insights

1. **Gender and Age**:
   - The correlation between Gender and Age is 0.1576, indicating a weak positive relationship.

2. **Vehicle Age and Other Variables**:
   - The strongest correlation observed is between Vehicle_Age and Previously_Insured (-0.3968). This negative correlation suggests that as the vehicle age increases, the likelihood of it being previously insured decreases.
   - There is a moderate positive correlation between Vehicle_Age and Response (0.1221), indicating older vehicles are somewhat more likely to have a response.

3. **Vehicle Damage**:
   - Vehicle_Damage has a strong negative correlation with Previously_Insured (-0.8362). This means that vehicles with damage are much less likely to have been previously insured.
   - It also has a moderate positive correlation with Response (0.3598), suggesting that vehicles with damage are more likely to have a response.

4. **Previously Insured**:
   - Previously_Insured has a strong negative correlation with Vehicle_Damage (-0.8362) and a moderate negative correlation with Response (-0.3453).

5. **Driving License**:
   - Driving_License shows negligible correlations with most other variables, indicating it is fairly independent in this dataset.

6. **Annual Premium**:
   - Annual_Premium has weak correlations with all other variables, the highest being with Age (0.0563).

7. **Policy Sales Channel and Vintage**:
   - Policy_Sales_Channel has weak correlations with other variables, the highest being with Previously_Insured (0.2368).
   - Vintage shows negligible correlation with most other variables, indicating it doesn't strongly associate with other aspects in the dataset.

### Main Observations

- Most correlations are weak, suggesting that the variables in the dataset are largely independent of one another.
- The notable exceptions are the correlations involving Vehicle_Age, Previously_Insured, and Vehicle_Damage, which show moderate to strong relationships.

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Exploring box-plot</b></div>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('dark_background')

sns.set(style="darkgrid")

columns_to_plot = [
    'Gender', 'Age', 'Driving_License', 'Region_Code', 'Previously_Insured',
    'Vehicle_Age', 'Vehicle_Damage', 'Annual_Premium', 'Policy_Sales_Channel',
    'Vintage', 'Response'
]

fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 20))

axes = axes.flatten()

for i, col in enumerate(columns_to_plot):
    sns.boxplot(x=train_df[col], ax=axes[i])
    axes[i].set_title(f'Box plot of {col}', fontsize=14, color='white')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()



### Here are some insights derived from the box plots: 

1. **Region_Code:**
   - **Insight:** The region codes are well-distributed across the range from 0 to 50. This suggests a diverse customer base from various regions without significant regional concentration.

2. **Previously_Insured:**
   - **Insight:** The binary distribution indicates that a considerable proportion of customers do not have prior insurance. This could be a target segment for marketing new insurance policies.

3. **Vehicle_Age:**
   - **Insight:** Vehicles are fairly evenly distributed across the three age categories. This suggests a mix of new, moderately used, and older vehicles, which may require different insurance products.

4. **Vehicle_Damage:**
   - **Insight:** The binary nature and balanced distribution suggest that vehicle damage history is a significant factor for a substantial portion of customers. Insurance companies might use this data for risk assessment.

5. **Annual_Premium:**
   - **Insight:** The right-skewed distribution with outliers indicates that while most customers pay a relatively low premium, a few pay significantly higher amounts. These outliers could represent high-value customers or those with high-risk profiles.

6. **Policy_Sales_Channel:**
   - **Insight:** The relatively uniform distribution suggests that multiple sales channels are being utilized effectively. Insurance companies should continue to support a diverse range of sales channels to reach different customer segments.

7. **Gender:**
   - **Insight:** The relatively balanced gender distribution indicates that the insurance products appeal equally to both males and females. Marketing strategies should, therefore, be inclusive and cater to both genders.

8. **Age:**
   - **Insight:** The age distribution is centered around middle-aged customers, with a slight skew towards older customers. Insurance companies could focus on products and services tailored to this age group, while also considering the needs of younger and older customers.

9. **Driving_License:**
   - **Insight:** The majority of customers have a driving license, which is expected for vehicle insurance customers. However, the presence of some customers without a driving license might indicate potential policyholders who are new drivers or those in the process of obtaining a license.

10. **Vintage:**
   - **Insight:** The vintage distribution suggests that customers have been with the insurance company for varying lengths of time, from new to long-term customers. Retention strategies should be developed for long-term customers while also creating attractive offers for new customers.

11. **Response:**
   - **Insight:** The target variable shows a strong imbalance with most customers not interested in the product (0), but a smaller segment is interested (1). This imbalance indicates a need for targeted marketing and personalized offers to convert the non-interested customers.


The presence of outliers in **Annual_Premium** suggests that we need to consider data preprocessing steps like scaling or transforming this feature to improve model performance.


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Exploring outliers through Z-Plot</b></div>

In [ ]:
from scipy import stats
import numpy as np


train_df['Annual_Premium_Z'] = stats.zscore(train_df['Annual_Premium'])

plt.figure(figsize=(10, 6))
sns.scatterplot(data=train_df, x='Age', y='Annual_Premium_Z')
plt.axhline(y=3, color='r', linestyle='--')
plt.axhline(y=-3, color='r', linestyle='--')
plt.title('Z-score plot of Annual Premium by Age')
plt.xlabel('Age')
plt.ylabel('Z-score of Annual Premium')
plt.show()

#### Now let us drop this extra Annual Premium Z-score column cauz why takig extra load lol

In [ ]:
train_df = train_df.drop(columns=['Annual_Premium_Z'])

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Checking For duplicate Column and Row values in the dataset</b></div>

In [ ]:
def get_duplicate_columns(df):
    duplicate_columns = {}
    seen_columns = {}
    for column in df.columns:
        current_column = df[column]
        # Convert column data to bytes
        try:
            current_column_hash = current_column.values.tobytes()
        except AttributeError:
            current_column_hash = current_column.to_string().encode()
        if current_column_hash in seen_columns:
            if seen_columns[current_column_hash] in duplicate_columns:
                duplicate_columns[seen_columns[current_column_hash]].append(column)
            else:
                duplicate_columns[seen_columns[current_column_hash]] = [column]
        else:
            seen_columns[current_column_hash] = column
    return duplicate_columns

In [ ]:
duplicate_columns = get_duplicate_columns(train_df)
duplicate_columns

In [ ]:
import pandas as pd

def check_duplicates(df, subset=None, remove=False):
    duplicates = df.duplicated(subset=subset)
    duplicate_rows = df[duplicates]
    print("Duplicate Rows:")
    print(duplicate_rows)
    duplicate_count = duplicates.sum()
    print(f"Number of duplicate rows: {duplicate_count}")
    
    if remove:
        df_no_duplicates = df.drop_duplicates(subset=subset)
        return df_no_duplicates, duplicate_count
    else:
        return df, duplicate_count


df_checked, duplicate_count = check_duplicates(train_df)
df_checked_specific, duplicate_count_specific = check_duplicates(train_df, subset=['Gender', 'Age', 'Driving_License', 'Region_Code', 'Previously_Insured', 'Vehicle_Age', 'Vehicle_Damage', 'Annual_Premium', 'Policy_Sales_Channel', 'Vintage', 'Response'], remove=True)
print("DataFrame with duplicates removed based on 'id' and 'name':")


#### Ah we got no duplicate column and rows lmao seems so perfect but anyways lets move forawrd

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Strategy for memory optimization using Klib</b></div>

To optimize the memory usage of our data processing pipeline, we implement a function to convert data types to more memory-efficient alternatives. This conversion is crucial in handling large datasets, especially when working with resource-constrained environments. By selecting appropriate data types, we can significantly reduce memory usage without sacrificing data integrity or precision.

We adhere to the following guidelines for data type conversion in Pandas to avoid data loss:

- int8: Suitable for integer values ranging from -128 to 127. Ideal for small categorical data.
- int16: Suitable for integer values ranging from -32,768 to 32,767. Useful for larger categorical data.
- int32: Suitable for integer values ranging from -2,147,483,648 to 2,147,483,647. Commonly used for general integer data.
- int64: Suitable for integer values ranging from -9,223,372,036,854,775,808 to 9,223,372,036,854,775,807. Used for large integer data that exceeds the range of int32.
- float16 (Half-precision): Provides an approximate decimal precision of 3 to 4 decimal digits. It is memory-efficient but has limited precision, making it suitable for less precise calculations.
- float32 (Single-precision): Provides an approximate decimal precision of 7 to 9 decimal digits. This is a balanced choice for most numerical computations.
- float64 (Double-precision): Provides an approximate decimal precision of 15 to 17 decimal digits. This type is used for high-precision calculations.
By converting data to these more efficient types, we ensure that our operations are not only faster but also more efficient in terms of memory usage. Here’s how you can implement such a conversion function:

In [ ]:
train_df.head()

#### Initially I tried doing it by writing a function but eventually I realised that there is a whole new class in Klib which makes things easy

In [ ]:
#def optimize_dtypes(df):
#    for col in df.columns:
#        col_type = df[col].dtype

#        if col_type != object:
#            if 'int' in str(col_type):
#                if df[col].min() >= -128 and df[col].max() <= 127:
#                    df[col] = df[col].astype('int8')
#                elif df[col].min() >= -32768 and df[col].max() <= 32767:
#                    df[col] = df[col].astype('int16')
#                elif df[col].min() >= -2147483648 and df[col].max() <= 2147483647:
#                    df[col] = df[col].astype('int32')
#                else:
#                    df[col] = df[col].astype('int64')
#            elif 'float' in str(col_type):
#                if df[col].apply(lambda x: len(str(x).split('.')[1]) if '.' in str(x) else 0).max() <= 4:
#                    df[col] = df[col].astype('float16')
#                elif df[col].apply(lambda x: len(str(x).split('.')[1]) if '.' in str(x) else 0).max() <= 9:
#                    df[col] = df[col].astype('float32')
#                else:
#                    df[col] = df[col].astype('float64')
    
#    return df


#train = optimize_dtypes(train_df)
#print(train.dtypes)


In [ ]:
import klib
train= klib.data_cleaning(train_df)

### As you can see without affecting the dataset we reduced its size by 82.5% so this is how powerful it is for largw datasets so now let us compare both train_df and train

In [ ]:
train_df.info()

In [ ]:
train_df.head()

In [ ]:
train.info()

In [ ]:
train.head()

#### Only problem with klib data_cleaning is it changes the feature names to smallrcase letters for space optimization so we need to rename it

In [ ]:
new_column_names = {
    'gender': 'Gender',
    'age': 'Age',
    'driving_license': 'Driving_License',
    'region_code': 'Region_Code',
    'previously_insured': 'Previously_Insured',
    'vehicle_age': 'Vehicle_Age',
    'vehicle_damage': 'Vehicle_Damage',
    'annual_premium': 'Annual_Premium',
    'policy_sales_channel': 'Policy_Sales_Channel',
    'vintage': 'Vintage',
    'response': 'Response'
}
train.rename(columns=new_column_names, inplace=True)

train.head()

In [ ]:
print(f'train_df dataFrame size: {train_df.shape}')
print(f'train dataFrame size: {train.shape}')

In [ ]:
from sklearn.model_selection import train_test_split

y = train['Response']
X = train.drop(columns=['Response'])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, shuffle=True, test_size=0.3)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Scaling our dataset</b></div>

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled= pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled= pd.DataFrame(X_test_scaled, columns=X_test.columns)
X_train_scaled.head()

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Feature importance using Random forest</b></div>

In [ ]:
# Let's look at the feature importance:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
fig = go.Figure(go.Bar(
            x=rf.feature_importances_,
            y=X_train.columns,
            orientation='h', marker_color='steelblue'))
fig.update_layout(template='plotly_dark', title='<b>Estimating feature importance through the Random Forest model', title_x=0.5, 
                 xaxis_title="Feature importance", yaxis_title='Feature')
fig.show()

### As expected we got the output kernel more than 100MB so i had to delete it :) but anyways What I found out is all features are imporant so cant remove any 

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Model Selection: Trying out different models in default settings and plotting it</b></div>

#### So in this part we try out all models and select the one with highest accuracy and lowest difference between crossval and train accuracy

Note: The output cell was more than 100MB so i had to delete the output :)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

df_models = pd.DataFrame(columns=['Algorithm', 'accuracy_train', 'accuracy_test'])

def make_model(X_tr, X_te, y_tr, y_te, model, model_name: str):
    model.fit(X_tr, y_tr)
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)
    acc_train = accuracy_score(y_tr, y_pred_train)
    acc_test = accuracy_score(y_te, y_pred_test)
    df_models.loc[len(df_models.index)] = [model_name, acc_train, acc_test]

make_model(X_train, X_test, y_train, y_test, GradientBoostingClassifier(), 'GradientBoosting')
make_model(X_train, X_test, y_train, y_test, XGBClassifier(tree_method='gpu_hist'), 'XGBoost')
make_model(X_train, X_test, y_train, y_test, lgb.LGBMClassifier(device='gpu'), 'LightGBM')
make_model(X_train, X_test, y_train, y_test, CatBoostClassifier(task_type='GPU'), 'CatBoost')



fig = go.Figure(data=[
    go.Bar(name='r2_train', x=df_models.Algorithm, y=df_models.r2_train),
    go.Bar(name='r2_test', x=df_models.Algorithm, y=df_models.r2_test)
])
fig.update_layout(template='plotly_dark', title='R2 for train and test', title_x=0.5)

#### As you cant see ouput cell here so basically I got XGboost and Catboost as the best performing model so let us more forward with XGBoost

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Optuna Class for XGBoost</b></div>

### Now let us do some hyperparameter tuning for this XGBoost class

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score


def objective(trial):
    param = {
        'tree_method': 'gpu_hist',
        'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
        'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]),
        'subsample': trial.suggest_categorical('subsample', [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.008, 0.01, 0.012, 0.014, 0.016, 0.018, 0.02]),
        'n_estimators': trial.suggest_int('n_estimators', 5, 1000),
        'max_depth': trial.suggest_categorical('max_depth', [5, 7, 9, 11, 13, 15, 17]),
        'random_state': trial.suggest_categorical('random_state', [2020]),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 300),
    }
    ratio = float(y_train.value_counts()[0]) / y_train.value_counts()[1]
    model = XGBClassifier(**param, scale_pos_weight=ratio)
    model.fit(X_train, y_train, eval_set=[(X_test_scaled, y_test)], early_stopping_rounds=100, verbose=True)
    y_pred = model.predict_proba(X_test)[:, 1]
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy


#### By mistake I made verbose=True so you need to skip some the output :') and Cant run the cell back again as it took roughly 1hr to run :)
#### Also I tried running it for different verbose and differnt value for different sets of values inside ```trial``` because of course this is a competition notebook so cant expose the best hyperparameters along with best model (I already exposed the best model) :/ so yes I run it twice which was really a pain to wait for this long :) but anyways

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Getting the Best parameter</b></div>

### As mentioned before this is not the best hyperparameter as I changed few stuffs so if you are trying to copy dont (It might perform well too on test dataset who knows XD)

In [ ]:
best_params = study.best_params
best_score = study.best_value
print(f"Best Hyperparameters: {best_params}")
print(f"Best Accuracy: {best_score:.3f}")

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Final training</b></div>
##### (Once again these are not the real params I used, yes I am clever XD)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

best_lambda = best_params['lambda']
best_alpha = best_params['alpha']
colsample_bytree = best_params['colsample_bytree']
subsample = best_params['subsample']
learning_rate = best_params['learning_rate']
max_depth = best_params['max_depth']
random_state = best_params['random_state']
min_child_weight = best_params['min_child_weight']
n_estimators = best_params['n_estimators']

best_model = XGBClassifier(
    tree_method='gpu_hist',  
    reg_lambda=best_lambda,
    alpha=best_alpha,
    colsample_bytree=colsample_bytree,
    subsample=subsample,
    learning_rate=learning_rate,
    max_depth=max_depth,
    random_state=random_state,
    min_child_weight=min_child_weight,
    n_estimators=n_estimators
)

best_model.fit(X_train, y_train)

In [ ]:
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.4f}")

In [ ]:
print(f"Validation Area Under the Curve (AUC): {roc_auc_score(y_test, y_pred)}")

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Transforming test dataset for submission</b></div>

#### Applying same feature transformation on Test

In [ ]:
test=pd.read_csv('/kaggle/input/playground-series-s4e7/test.csv')
test.head()

In [ ]:
test_ids = test['id']
test.drop('id', axis=1, inplace=True)
test.head()

In [ ]:
def convert_vehicle_age(age_str):
    if isinstance(age_str, str):
        if '< 1 Year' in age_str:
            return 0
        elif '1-2 Year' in age_str:
            return 1
        elif '> 2 Years' in age_str:
            return 2
    return None

test['Vehicle_Age'] = test['Vehicle_Age'].apply(convert_vehicle_age)
test.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

test['Gender'] = label_encoder.fit_transform(test['Gender'])
test['Vehicle_Damage'] = label_encoder.fit_transform(test['Vehicle_Damage'])
test.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(test)
test_scaled = scaler.transform(test)
test_scaled= pd.DataFrame(test_scaled, columns=test.columns)

In [ ]:
print("Shape of train_df:", X_train.shape)
print("Shape of test:", test_scaled.shape)

In [ ]:
predictions_test = best_model.predict_proba(test_scaled)[:, 1]

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Submission Time 🥳</b></div>

In [ ]:
result = pd.DataFrame({
    'id': test_ids,
    'Response': predictions_test.flatten()  
}, columns=['id', 'Response'])

result.to_csv("/kaggle/working/submission.csv", index=False)

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Sneaking how Optuna performed</b></div>

In [ ]:
#plotting the optimization history of the study
optuna.visualization.plot_optimization_history(study)

In [ ]:
#studying the relationship between each value of hyperparameter through this plot
optuna.visualization.plot_parallel_coordinate(study)

In [ ]:
# now we plot the accuracy of each hyperparameter for each trial
optuna.visualization.plot_slice(study)

In [ ]:
#plotting the accuracy surface for the hyperparameter involved in the xgb
optuna.visualization.plot_contour(study, params=['alpha',
                            'lambda',
                            'subsample',
                            'learning_rate','subsample','n_estimators'])

In [ ]:
#Now we look at the role of each parameter in this process
optuna.visualization.plot_param_importances(study)